# Human Gene Essentiality Predictor V2
## Hedge Fund + Expression Features for Context-Dependent Genes

**V1 Results:**
- Pan-essential/Non-essential: 98-100% accuracy ✅
- Context-dependent genes: 43% accuracy ❌

**V2 Goal:** Add expression features to improve context-dependent prediction

## 1. Setup & Data Loading

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

In [ ]:
# ============================================================
# UPDATE THESE PATHS TO YOUR DRIVE LOCATIONS
# ============================================================
DEPMAP_PATH = '/content/drive/MyDrive/depmap_data/CRISPRGeneEffect.csv'
EXPRESSION_PATH = '/content/drive/MyDrive/depmap_data/OmicsExpressionProteinCodingGenesTPMLogp1.csv'
# Optional: Model metadata for cancer type info
# MODEL_PATH = '/content/drive/MyDrive/depmap_data/Model.csv'

In [ ]:
# Load CRISPR essentiality data
print("Loading CRISPR gene effect data...")
crispr_df = pd.read_csv(DEPMAP_PATH, index_col=0)
print(f"CRISPR shape: {crispr_df.shape} (cell_lines x genes)")

# Load expression data
print("\nLoading expression data...")
expr_df = pd.read_csv(EXPRESSION_PATH, index_col=0)
print(f"Expression shape: {expr_df.shape} (cell_lines x genes)")

In [ ]:
# Find overlapping cell lines and genes
common_cell_lines = list(set(crispr_df.index) & set(expr_df.index))
print(f"Common cell lines: {len(common_cell_lines)}")

# Gene names might have different formats - let's check
print(f"\nCRISPR gene example: {crispr_df.columns[0]}")
print(f"Expression gene example: {expr_df.columns[0]}")

In [ ]:
# Standardize gene names (extract gene symbol before parentheses if needed)
def extract_gene_name(col):
    """Extract gene name from column like 'TP53 (7157)' -> 'TP53'"""
    if ' (' in col:
        return col.split(' (')[0]
    return col

# Create mapping
crispr_gene_map = {col: extract_gene_name(col) for col in crispr_df.columns}
expr_gene_map = {col: extract_gene_name(col) for col in expr_df.columns}

# Rename columns
crispr_renamed = crispr_df.rename(columns=crispr_gene_map)
expr_renamed = expr_df.rename(columns=expr_gene_map)

# Find common genes
common_genes = list(set(crispr_renamed.columns) & set(expr_renamed.columns))
print(f"Common genes: {len(common_genes)}")

In [ ]:
# Filter to common cell lines and genes
crispr_filtered = crispr_renamed.loc[common_cell_lines, common_genes]
expr_filtered = expr_renamed.loc[common_cell_lines, common_genes]

print(f"Filtered CRISPR: {crispr_filtered.shape}")
print(f"Filtered Expression: {expr_filtered.shape}")

# Transpose so genes are rows
crispr_mat = crispr_filtered.T
expr_mat = expr_filtered.T

print(f"\nFinal matrices: {crispr_mat.shape[0]} genes x {crispr_mat.shape[1]} cell lines")

## 2. Binarize & Classify Genes

In [ ]:
# Binarize CRISPR scores (negative = essential)
THRESHOLD = -0.5
binary_mat = (crispr_mat < THRESHOLD).astype(int)

# Calculate consensus
gene_consensus = binary_mat.mean(axis=1)

# Classify genes
gene_category = pd.Series(index=binary_mat.index, dtype=str)
gene_category[gene_consensus >= 0.9] = 'pan_essential'
gene_category[(gene_consensus >= 0.5) & (gene_consensus < 0.9)] = 'common_essential'
gene_category[(gene_consensus >= 0.1) & (gene_consensus < 0.5)] = 'context_dependent'
gene_category[(gene_consensus > 0) & (gene_consensus < 0.1)] = 'rarely_essential'
gene_category[gene_consensus == 0] = 'non_essential'

print("Gene categories:")
print(gene_category.value_counts())

## 3. V2 Hedge Fund Predictor with Expression Features

In [ ]:
class HedgeFundPredictorV2:
    """
    V2: Adds expression features for context-dependent genes.
    
    Key insight: Context-dependent genes are often essential when 
    HIGHLY EXPRESSED in a cell line (the cell depends on them).
    """
    
    def __init__(self, high_thresh=0.9, low_thresh=0.1):
        self.high_thresh = high_thresh
        self.low_thresh = low_thresh
        self.ml_model = GradientBoostingClassifier(
            n_estimators=100, 
            max_depth=5,
            learning_rate=0.1,
            random_state=42
        )
        self.scaler = StandardScaler()
        self.is_fitted = False
        
    def _get_features(self, binary_mat, expr_mat, gene, cell_line, other_cols):
        """
        Extract features for a (gene, cell_line) pair.
        
        Features:
        1. Consensus essentiality (from other cell lines)
        2. Variance in essentiality
        3. Expression level of this gene in this cell line
        4. Expression percentile (relative to other cell lines)
        5. Expression z-score
        """
        # Consensus features (from other cell lines)
        consensus = binary_mat.loc[gene, other_cols].mean()
        variance = binary_mat.loc[gene, other_cols].var()
        
        # Expression features
        expr_value = expr_mat.loc[gene, cell_line]
        expr_others = expr_mat.loc[gene, other_cols]
        expr_mean = expr_others.mean()
        expr_std = expr_others.std() + 1e-6  # Avoid division by zero
        expr_zscore = (expr_value - expr_mean) / expr_std
        expr_percentile = (expr_others < expr_value).mean()
        
        return [
            consensus,
            variance,
            consensus ** 2,
            expr_value,
            expr_zscore,
            expr_percentile,
            expr_value * consensus,  # Interaction term
        ]
    
    def fit(self, binary_mat, expr_mat, context_genes, sample_size=50000):
        """
        Train the ML model on context-dependent genes.
        """
        print(f"Training on {len(context_genes)} context-dependent genes...")
        
        X_train = []
        y_train = []
        
        cell_lines = list(binary_mat.columns)
        
        # Sample training data (full data might be too large)
        pairs = [(g, c) for g in context_genes for c in cell_lines]
        if len(pairs) > sample_size:
            np.random.seed(42)
            indices = np.random.choice(len(pairs), sample_size, replace=False)
            pairs = [pairs[i] for i in indices]
        
        print(f"Sampling {len(pairs)} (gene, cell_line) pairs for training...")
        
        for gene, cell_line in tqdm(pairs, desc="Extracting features"):
            other_cols = [c for c in cell_lines if c != cell_line]
            features = self._get_features(binary_mat, expr_mat, gene, cell_line, other_cols)
            X_train.append(features)
            y_train.append(binary_mat.loc[gene, cell_line])
        
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        
        # Handle NaN/Inf
        X_train = np.nan_to_num(X_train, nan=0, posinf=0, neginf=0)
        
        # Scale and fit
        X_train_scaled = self.scaler.fit_transform(X_train)
        self.ml_model.fit(X_train_scaled, y_train)
        
        # Report training accuracy
        y_pred = self.ml_model.predict(X_train_scaled)
        print(f"Training Accuracy: {accuracy_score(y_train, y_pred):.3f}")
        print(f"Training Balanced Accuracy: {balanced_accuracy_score(y_train, y_pred):.3f}")
        
        # Feature importance
        feature_names = ['consensus', 'variance', 'consensus^2', 'expression', 
                        'expr_zscore', 'expr_percentile', 'expr*consensus']
        importances = self.ml_model.feature_importances_
        print("\nFeature Importances:")
        for name, imp in sorted(zip(feature_names, importances), key=lambda x: -x[1]):
            print(f"  {name:20s}: {imp:.3f}")
        
        self.is_fitted = True
        return self
    
    def predict_cell_line(self, binary_mat, expr_mat, gene_category, held_out_col):
        """
        Predict essentiality for one held-out cell line.
        """
        other_cols = [c for c in binary_mat.columns if c != held_out_col]
        consensus = binary_mat[other_cols].mean(axis=1)
        
        predictions = pd.Series(index=binary_mat.index, dtype=int)
        
        # Pan-essential: always essential
        predictions[gene_category == 'pan_essential'] = 1
        
        # Non-essential: never essential
        predictions[gene_category == 'non_essential'] = 0
        
        # Rarely essential: use threshold
        rarely_mask = gene_category == 'rarely_essential'
        predictions[rarely_mask] = (consensus[rarely_mask] >= 0.05).astype(int)
        
        # Context-dependent & common: use ML with expression
        ml_mask = (gene_category == 'context_dependent') | (gene_category == 'common_essential')
        ml_genes = predictions[ml_mask].index.tolist()
        
        if len(ml_genes) > 0 and self.is_fitted:
            X_test = []
            for gene in ml_genes:
                features = self._get_features(binary_mat, expr_mat, gene, held_out_col, other_cols)
                X_test.append(features)
            
            X_test = np.array(X_test)
            X_test = np.nan_to_num(X_test, nan=0, posinf=0, neginf=0)
            X_test_scaled = self.scaler.transform(X_test)
            
            ml_pred = self.ml_model.predict(X_test_scaled)
            for i, gene in enumerate(ml_genes):
                predictions[gene] = ml_pred[i]
        
        return predictions

print("HedgeFundPredictorV2 defined!")

## 4. Train the Model

In [ ]:
# Get context-dependent and common essential genes (where ML is needed)
ml_genes = gene_category[(gene_category == 'context_dependent') | 
                          (gene_category == 'common_essential')].index.tolist()

print(f"Genes for ML training: {len(ml_genes)}")

# Initialize and train predictor
predictor_v2 = HedgeFundPredictorV2()
predictor_v2.fit(binary_mat, expr_mat, ml_genes, sample_size=100000)

## 5. Evaluate with Leave-One-Out CV

In [ ]:
# Evaluate on a sample of cell lines
N_SAMPLES = 50  # Increase for more thorough evaluation

cell_lines = list(binary_mat.columns)
np.random.seed(42)
test_cell_lines = np.random.choice(cell_lines, min(N_SAMPLES, len(cell_lines)), replace=False)

all_y_true = []
all_y_pred = []
all_categories = []

print(f"Evaluating on {len(test_cell_lines)} cell lines...")
for cell_line in tqdm(test_cell_lines):
    y_true = binary_mat[cell_line].values
    y_pred = predictor_v2.predict_cell_line(binary_mat, expr_mat, gene_category, cell_line).values
    
    all_y_true.extend(y_true)
    all_y_pred.extend(y_pred)
    all_categories.extend(gene_category.values)

y_true = np.array(all_y_true)
y_pred = np.array(all_y_pred)
categories = np.array(all_categories)

print("\n" + "="*60)
print("OVERALL RESULTS")
print("="*60)
print(f"Accuracy:          {accuracy_score(y_true, y_pred):.3f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_true, y_pred):.3f}")
print(f"F1 Score:          {f1_score(y_true, y_pred):.3f}")

In [ ]:
# Results by category
print("\n" + "="*60)
print("RESULTS BY GENE CATEGORY")
print("="*60)

results = []
for cat in ['pan_essential', 'common_essential', 'context_dependent', 'rarely_essential', 'non_essential']:
    mask = categories == cat
    if mask.sum() > 0:
        acc = accuracy_score(y_true[mask], y_pred[mask])
        bal_acc = balanced_accuracy_score(y_true[mask], y_pred[mask])
        n = mask.sum()
        results.append({'Category': cat, 'Accuracy': acc, 'Balanced_Acc': bal_acc, 'N': n})
        print(f"{cat:25s}: Acc={acc:.3f}, Bal_Acc={bal_acc:.3f} (n={n:,})")

results_df = pd.DataFrame(results)

In [ ]:
# Visualize comparison with V1
v1_acc = [0.982, 0.434, 0.439, 0.991, 1.0]  # From your previous run
v2_acc = results_df['Accuracy'].values

x = np.arange(len(results_df))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, v1_acc, width, label='V1 (Consensus Only)', color='#ff6b6b', alpha=0.8)
bars2 = ax.bar(x + width/2, v2_acc, width, label='V2 (+ Expression)', color='#4ecdc4', alpha=0.8)

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
ax.set_ylabel('Accuracy')
ax.set_title('V1 vs V2: Impact of Expression Features')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Category'], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)

# Add value labels
for bar, val in zip(bars1, v1_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val:.1%}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, v2_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val:.1%}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix for context-dependent genes only
context_mask = (categories == 'context_dependent') | (categories == 'common_essential')
cm = confusion_matrix(y_true[context_mask], y_pred[context_mask])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Essential', 'Essential'],
            yticklabels=['Non-Essential', 'Essential'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Context-Dependent Genes (V2 with Expression)')
plt.show()

## 6. Analyze Expression-Essentiality Relationship

In [ ]:
# For context-dependent genes: how does expression relate to essentiality?
context_genes = gene_category[gene_category == 'context_dependent'].index.tolist()[:100]

expr_when_essential = []
expr_when_not_essential = []

for gene in context_genes:
    for cell_line in binary_mat.columns[:200]:
        is_essential = binary_mat.loc[gene, cell_line]
        expr_value = expr_mat.loc[gene, cell_line]
        
        if is_essential == 1:
            expr_when_essential.append(expr_value)
        else:
            expr_when_not_essential.append(expr_value)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(expr_when_not_essential, bins=50, alpha=0.6, label='Non-Essential', density=True)
ax.hist(expr_when_essential, bins=50, alpha=0.6, label='Essential', density=True)
ax.set_xlabel('Expression Level (log2 TPM+1)')
ax.set_ylabel('Density')
ax.set_title('Expression Distribution: Essential vs Non-Essential\n(Context-Dependent Genes)')
ax.legend()
plt.show()

print(f"Mean expression when essential: {np.mean(expr_when_essential):.2f}")
print(f"Mean expression when not essential: {np.mean(expr_when_not_essential):.2f}")

## 7. Save Results

In [ ]:
# Save updated gene summary
gene_summary_v2 = pd.DataFrame({
    'gene': binary_mat.index,
    'consensus': gene_consensus.values,
    'category': gene_category.values,
    'mean_expression': expr_mat.mean(axis=1).values,
    'expression_variance': expr_mat.var(axis=1).values,
}).sort_values('consensus', ascending=False)

OUTPUT_PATH = '/content/drive/MyDrive/depmap_data/gene_essentiality_v2_with_expression.csv'
gene_summary_v2.to_csv(OUTPUT_PATH, index=False)
print(f"Results saved to: {OUTPUT_PATH}")

## 8. Summary

### V1 → V2 Comparison

| Category | V1 (Consensus Only) | V2 (+ Expression) | Improvement |
|----------|--------------------|--------------------|-------------|
| Pan-essential | 98.2% | ~98% | - |
| Common essential | 43.4% | **??%** | **+??%** |
| Context-dependent | 43.9% | **??%** | **+??%** |
| Rarely essential | 99.1% | ~99% | - |
| Non-essential | 100% | 100% | - |

### Key Insight
Expression level is predictive of essentiality for context-dependent genes:
- High expression → more likely essential (cell depends on it)
- Low expression → less likely essential (cell doesn't use it much)